# Dispatch in notebooks: a tutorial

Dispatch runs SQL against the Impala cluster from the Hadoop Edge Node. This
notebook is the tutorial for `dispatch.notebook`, the Python API: how to run a
query written in a cell, load its result into pandas, supervise long Jobs, and
stay out of trouble with resource pools, the two-Job cap, and the Advisor.

Work through it top to bottom; every cell runs.

## The one thing to understand first

**Every query you run here is a Job**, the same unit of work the `dispatch`
TUI and the `dispatch job` CLI create: it is validated, checked against your
Kerberos ticket, analysed by the Advisor, admitted against a two-Job limit,
written to a manifest on disk, and executed by a detached runner that survives
your kernel dying.

That makes this API unlike duckdb or a database cursor:

| | duckdb / cursor | Dispatch |
|---|---|---|
| Latency | milliseconds | minutes — queue cycling, real cluster |
| Concurrency | as many as you like | two Jobs Pending or Running, per user |
| Failure | an exception | a Job in state `Failed`, with a run log |
| Audit | none | manifest, run log, telemetry |
| Kernel restart | loses the query | Job keeps running, pick it back up |

So: no loops firing off dozens of queries, and no expectation of instant
answers. In exchange, a Job you start before lunch is still running, and still
inspectable, when you get back.

## Before you start

On the Edge Node you need a kernel that can `import dispatch` and a valid
Kerberos ticket (`kinit` in a terminal). If the kernel cannot import the
package, point the library at the installed launcher instead:
`export DISPATCH_CLI=~/.local/bin/dispatch` before starting Jupyter.

Locally, start the kernel from a shell that has sourced `mocks/dev-env.sh`, so
the fake `impala-shell`, `klist`, and SMTP catcher are on `PATH`. Everything
below then runs offline — but note that the mock `impala-shell` returns the
same two-column stub (`id,value`) for every query, so the *shape* of the
results here is not the shape of your real data.

## 1. Open a session

`Dispatch` is a handle, not a connection: nothing is held open, and each call
runs one `dispatch job` command.

- `cwd` is where your `.sql` files live and where CSVs you name by hand are
  written. It defaults to the notebook's working directory.
- `workspace` is Dispatch-owned storage for SQL you write in cells and the
  results it fetches, so this notebook never litters `cwd`.

In [ ]:
import time
from datetime import date
from pathlib import Path

import pandas as pd

from dispatch.notebook import (
    Dispatch,
    JobUnsuccessful,
    MissingResultError,
    OperationalError,
    ResultParseError,
    UsageError,
    WaitTimeout,
)

workdir = Path.cwd()
d = Dispatch(cwd=workdir)

print("cwd:      ", d.cwd)
print("workspace:", d.workspace)
print("cli:      ", " ".join(d.command))
d

A listing is the cheapest way to check the session works end to end: it runs
the real CLI, reconciles any stale manifests, and renders as a table. An empty
list means "no Jobs yet", not "broken".

In [ ]:
d.jobs()

## 2. Run SQL written in a cell

`d.sql(text)` saves your SQL into the workspace and launches it as a Job. It
returns as soon as the Job is handed to the runner — it does **not** wait — so
the object you get back is a `Job`, not rows.

In [ ]:
job = d.sql(
    """
    SELECT dt, count(*) AS events
    FROM aa_enc.events
    WHERE dt = '2026-07-01'
    GROUP BY dt
    """
)
job

Two things to notice.

The Job has an ID (`20260728T...`) that is stable forever — write it down and
you can find this Job from any notebook, the CLI, or the TUI.

Submission is eager. There is no lazy query object to chain `.filter()` onto,
because Dispatch has no planner to optimise: it hands your SQL to Impala as
written. The upside is that mistakes surface here, at the line that made them,
rather than later.

In [ ]:
print("id:         ", job.id)
print("state:      ", job.state)
print("source:     ", job.source, job.source_detail)
print("destination:", job.destination, job.destination_detail)
print("queue:      ", job.params["queue"])

## 3. Get the rows into pandas

`to_df()` waits for the Job to finish, then reads the CSV it exported. Use
`to_pandas()` if that name is more familiar; they are the same method.

In [ ]:
frame = job.to_df(poll_interval=1.0)
print(job.state, "in", job.elapsed_seconds, "s")
frame

Other ways to reach the same result:

| Call | Gives you |
|---|---|
| `job.to_df()` | a `DataFrame`, dtypes inferred by pandas |
| `job.rows()` | `list[dict[str, str]]`, every value a string |
| `job.columns` | the column names |
| `job.result_path` | the CSV on disk |
| `job.to_csv(path)` | a copy wherever you want it |

`to_df()` forwards keyword arguments to `pandas.read_csv`, which is how you
take control of typing: `dtype=`, `parse_dates=`, `nrows=`, `usecols=`.

In [ ]:
print("columns:", job.columns)
print("rows:   ", job.rows())
print("csv:    ", job.result_path)
print("dtypes: ", dict(job.to_df().dtypes.astype(str)))
print("as text:", dict(job.to_df(dtype="string").dtypes.astype(str)))

Results live in the workspace and are yours to keep or discard. When a result
is a deliverable, copy it out by name — that is the file you would email or
hand to someone.

In [ ]:
kept = job.to_csv(workdir / "july_events.csv")
print(kept, "->", kept.read_text(encoding="utf-8").splitlines()[:2])

### One sharp edge worth knowing

The export is written by `impala-shell` in delimited mode, which **does not
quote fields**. A string value containing a comma or a newline therefore
produces an ambiguous CSV, and pandas will not tell you: a line with too many
fields silently becomes an index column, and a line with too few is padded
with `NaN`.

Dispatch refuses to guess. Both readers check every line against the header
and raise `ResultParseError` naming the line. The cell below corrupts a result
on purpose to show the error you would get.

In [ ]:
good = job.result_path.read_text(encoding="utf-8")
job.result_path.write_text("id,value\n1,oops,extra\n", encoding="utf-8")

try:
    job.rows()
except ResultParseError as exc:
    print(f"ResultParseError: {exc}")

job.result_path.write_text(good, encoding="utf-8")
print("\nrestored:", job.rows())

The fix belongs in SQL, not in pandas: strip the delimiter from the offending
column, for example `regexp_replace(description, ',', ' ') AS description`.
For a very large result you can skip the integrity scan with
`job.to_df(strict=False)` and accept pandas' behaviour.

## 4. Read a table that already exists

`d.table("schema.table")` exports a table without you writing any SQL. Always
pass `limit=` unless you truly want the whole table: without it the export is
`select * from <table>` with no bound, which on a real table can be enormous.

In [ ]:
peek = d.table("aa_enc.events_existing", limit=10)
peek.to_df(poll_interval=1.0)

With `limit=` Dispatch generates `SELECT * FROM <table> LIMIT <n>` and runs it
as ordinary SQL, so the Advisor sees it like anything else. That means an
unfiltered peek at a monitored schema (`core`, `gco`, `mrs`) can be refused —
see section 8 — while the unbounded whole-table export cannot be, because
there is no SQL text to analyse. Prefer the bounded form anyway.

## 5. Watch a Job while it runs

Real Jobs take minutes. `watch()` refreshes the state and the tail of the run
log in place until the Job reaches a terminal state.

Locally the mocks finish almost immediately, so you will see one frame. Export
`DISPATCH_MOCK_DELAY=15` before starting Jupyter to watch it tick.

In [ ]:
watched = d.sql("SELECT 1 AS one")
watched.watch(poll_interval=1.0, lines=8)

When you would rather not sit and stare:

| Call | Behaviour |
|---|---|
| `job.wait(timeout=600)` | block until terminal; `WaitTimeout` if the deadline passes |
| `job.refresh()` | one poll, no waiting |
| `job.logs(lines=100)` | the last N log lines as text |
| `job.print_logs(follow=True)` | stream the log until the Job ends |
| `job.stream_logs()` | the same stream as an iterator |

State is data, not an exception: a Job that ran and failed is returned like any
other. Read `state`, `succeeded`, `failed`, `cancelled`, `exit_code`,
`elapsed_seconds`.

In [ ]:
print("terminal:", watched.is_terminal, "| succeeded:", watched.succeeded)
print("exit code:", watched.exit_code, "| elapsed:", watched.elapsed_seconds, "s")
print()
print(watched.logs(lines=4))

A timeout stops *watching*, never the Job. The runner is detached, so a Job that
outlives your patience carries on, and you can wait on it again — from this
notebook, another notebook, or `dispatch job wait` in a terminal. Section 6 shows
that with a deliberately slow Job.

In [ ]:
print("same Job, seen from a second handle:", d.job(watched.id))

## 6. Come back to a Job later

This is the payoff for every query being a Job. Restart the kernel, close the
laptop, hand the ID to a colleague: the Job is on disk, so `d.job(job_id)`
returns it with everything intact, including its Result.

In [ ]:
saved_id = job.id

later = Dispatch(cwd=workdir).job(saved_id)   # pretend this is a fresh kernel
print(later.state, later.result_path)
later.to_df()

The cell below starts a deliberately slow Job (a mock delay in local mode) so
there is something to be impatient about. Waiting with a short `timeout` raises
`WaitTimeout` — and the Job keeps running, because your deadline was about you,
not about it.

Cancelling is how you actually stop one. It signals the runner's whole process
group, so the Impala query stops too, and the Job ends in `Cancelled` rather
than vanishing from the record.

In [ ]:
slow = Dispatch(cwd=workdir, env={"DISPATCH_MOCK_DELAY": "20"})
doomed = slow.sql("SELECT 'this takes a while' AS note")

deadline = time.monotonic() + 20
while doomed.refresh().state == "Pending" and time.monotonic() < deadline:
    time.sleep(0.5)
print("state:", doomed.state, "| pid:", doomed.pid)

try:
    doomed.wait(timeout=1.0, poll_interval=0.5)
except WaitTimeout as exc:
    print(f"WaitTimeout: {exc}")
print("still running after the timeout:", doomed.refresh().state)

doomed.cancel()
print("after cancel:", doomed.state, "| exit code:", doomed.exit_code)

## 7. Destinations: rows, a table, or both

A Job has exactly one destination, and it decides what you get back.

| `destination` | Writes | Has a Result to read? |
|---|---|---|
| `"Csv"` (default) | a CSV | yes |
| `"Table"` | a parquet Impala table | no |
| `"Table+Csv"` | both, table first | yes |

Materialise a table when the output is an input to something else — another
query, a dashboard, a colleague's Job. `table=` is required, because Dispatch
will not invent a name for a table that outlives your session. Your analyst ID
is prefixed automatically.

In [ ]:
materialised = d.sql(
    "SELECT dt, count(*) AS events FROM aa_enc.events GROUP BY dt",
    destination="Table",
    table="tutorial_daily_events",
).wait(poll_interval=1.0)

print(materialised.state, "->", materialised.destination_detail)

try:
    materialised.to_df()
except MissingResultError as exc:
    print(f"MissingResultError: {exc}")

`"Table+Csv"` gives you both in one Job: the table for later, the rows now.

In [ ]:
both = d.sql(
    "SELECT dt, count(*) AS events FROM aa_enc.events GROUP BY dt",
    destination="Table+Csv",
    table="tutorial_daily_events_csv",
)
both.to_df(poll_interval=1.0)

### Monthly jobs over a date range

For a query that should run once per month across a range, write the
`{date_inicio}` / `{date_fim}` placeholders and pass `source="SqlTemplate"`
with the range. Dispatch runs the template month by month into one table —
much kinder to the cluster than one enormous scan.

In [ ]:
monthly = d.sql(
    """
    SELECT '{date_inicio}' AS month_start,
           '{date_fim}'    AS month_end,
           count(*)        AS events
    FROM aa_enc.events
    WHERE dt BETWEEN '{date_inicio}' AND '{date_fim}'
    """,
    source="SqlTemplate",
    destination="Table",
    table="tutorial_monthly",
    start_date=date(2026, 1, 1),
    end_date=date(2026, 3, 31),
).wait(poll_interval=1.0)

print(monthly.state, monthly.params["start_date"], "->", monthly.params["end_date"])

## 8. When Dispatch says no

Refusals are exceptions; Job outcomes are data. Two families:

| Family | Members | Meaning |
|---|---|---|
| `DispatchError` | `UsageError`, `UnknownJobError`, `OperationalError`, `JobUnsuccessful`, `WaitTimeout` | a command was refused, or the Job did not succeed |
| `ResultError` | `MissingResultError`, `ResultParseError` | the Job ran, but its export cannot be read |

### The Advisor gate

Before launching, Dispatch analyses your SQL against the Impala optimisation
guidelines. Error-severity findings — a cartesian product, an unfiltered
`SELECT *` over a monitored table, a date range beyond 13 months — stop the
launch with a `UsageError` that names the rule. It is a speed bump, not a
wall: if you meant it, acknowledge it and Dispatch runs your SQL as written.

In [ ]:
cross_join = "SELECT a.x FROM aa_enc.t1 a CROSS JOIN aa_enc.t2 b"

try:
    d.sql(cross_join)
except UsageError as exc:
    print(f"refused (exit {exc.exit_code}): {exc}")

acknowledged = d.sql(cross_join, acknowledge_advisor=True)
print("launched anyway:", acknowledged.id)

### A Job that fails

Bad SQL is not refused at launch — Impala is the judge — so the Job runs and
ends in `Failed`. Nothing raises until you ask for rows that do not exist; then
`JobUnsuccessful` points you at the log, which is where the actual Impala error
lives.

In [ ]:
broken_session = Dispatch(cwd=workdir, env={"DISPATCH_MOCK_SCENARIO": "syntax_error"})
broken = broken_session.sql("SELECT FROM nowhere")

try:
    broken.to_df(poll_interval=1.0)
except JobUnsuccessful as exc:
    print(f"JobUnsuccessful: {exc}\n")

print(broken.state, "exit code", broken.exit_code)
print("\n".join(broken.logs(lines=6).splitlines()[-4:]))

Other refusals you will meet, and what they mean:

| Message | Cause | Fix |
|---|---|---|
| `Kerberos ticket missing - run kinit` | no ticket, or under 5 minutes left | `kinit` in a terminal, then rerun the cell |
| `two Dispatch jobs already occupy shared capacity` | the two-Job cap | wait, cancel one, or use `wait_for_slot` (section 10) |
| `SQL file not found` | a `d.launch(sql=...)` path that does not exist | check the path relative to `d.cwd` |
| `Malformed Job ID` | a typo'd ID | copy it from `d.jobs()` |
| `Could not run the Dispatch CLI` | the kernel cannot reach `dispatch` | set `DISPATCH_CLI` and restart the kernel |

## 9. Resource pools and retries

Impala queues work through **resource pools**. By default Dispatch leaves the
choice alone (`queue="auto"`) and the orchestrator cycles its own pool list,
trying the next pool whenever one is busy and waiting 30 seconds between full
cycles — indefinitely for CSV and table Jobs, up to ten cycles for monthly
ones. Fatal errors (syntax, table not found, auth) stop immediately instead of
cycling.

Pin pools when you know what you need: a small pool for a small query, a large
one for a heavy aggregation. Pass one name or a list, and Dispatch orders them
canonically (`adhoc_fast, adhoc_small, acs_small, acs_large, adhoc`).

In [ ]:
pinned = d.sql(
    "SELECT count(*) AS n FROM aa_enc.events",
    queue=["acs_large", "adhoc"],
).wait(poll_interval=1.0)

print("requested pools:", pinned.params["queue"])
print([line for line in pinned.logs(lines=40).splitlines() if "queue" in line][:3])

Two details that surprise people:

- Pinning a single pool turns retry into a one-pool loop that keeps trying that
  pool every 30 seconds. If it is congested, `auto` finishes sooner.
- `auto` is not the same list for every Job: CSV exports cycle
  `adhoc_fast, adhoc_small, adhoc`, while table Jobs also try the `acs_*`
  pools. Name pools explicitly if you care.

## 10. Two Jobs at a time

You may have two Jobs `Pending` or `Running` at once — a shared limit that
protects the cluster, counted across your notebooks, terminals, and the TUI.
A third launch is refused immediately with `OperationalError`.

Two Jobs at once is a feature, not just a limit: launch both, then collect.

In [ ]:
first = d.sql("SELECT 'first' AS which")
second = d.sql("SELECT 'second' AS which")

frames = {job.id: job.to_df(poll_interval=1.0) for job in (first, second)}
print(f"{len(frames)} results collected")
pd.concat(frames.values(), keys=frames.keys())

For a batch of more than two, `wait_for_slot=<seconds>` turns the cap into
backpressure: instead of failing, the launch retries until a slot frees or the
deadline passes. Only capacity refusals are retried — a Kerberos or validation
problem still surfaces at once, because retrying it would be pointless.

This is the shape of a batch loop. Nothing is queued behind your back: at most
two Jobs run, and the loop blocks on the third until one finishes.

In [ ]:
months = ["2026-01", "2026-02", "2026-03"]
batch = []

for month in months:
    batch.append(
        d.sql(
            f"SELECT '{month}' AS month, count(*) AS events "
            f"FROM aa_enc.events WHERE dt LIKE '{month}%'",
            wait_for_slot=600,
        )
    )

results = pd.concat(job.to_df(poll_interval=1.0) for job in batch)
print(f"{len(batch)} Jobs, {len(results)} rows")
results

## 11. Supervise everything you have running

`d.jobs()` lists Jobs newest-first after reconciling stale manifests, and
renders as a table here. It is the same view as `dispatch job list` in a
terminal and the dashboard in the TUI — one set of Jobs, three windows onto it.

In [ ]:
d.jobs()

In [ ]:
summary = pd.DataFrame(d.jobs().to_dicts())
summary["queue"] = summary["params"].apply(lambda params: params.get("queue"))

print(summary.groupby(["state", "destination"]).size().rename("jobs").to_string())
summary[["id", "state", "destination", "queue", "exit_code"]].head(6)

In [ ]:
print("running now:  ", [job.id for job in d.jobs(state="Running")])
print("failed today: ", [job.id for job in d.jobs(state="Failed")])
print("cancelled:    ", [job.id for job in d.jobs(state="Cancelled")])

## 12. Launch from a `.sql` file

Inline SQL is convenient; a file is better when the query is long, shared, or
version-controlled. `d.launch()` is the file-based form, and it exposes every
CLI flag: source, destination, schema, table, dates, email, subject, queue.

Its CSV lands in `cwd`, next to the SQL, because you named it — unlike inline
results, which go to the workspace.

In [ ]:
(workdir / "tutorial_query.sql").write_text(
    "SELECT dt, count(*) AS events FROM aa_enc.events GROUP BY dt\n",
    encoding="utf-8",
)

from_file = d.launch(
    source="SqlFile",
    destination="Csv",
    sql="tutorial_query.sql",
    table="tutorial_from_file",
    email="",              # orchestrators notify these addresses; empty means none
    subject="Tutorial run",
).wait(poll_interval=1.0)

print(from_file.state, "->", from_file.result_path)
from_file.to_df()

## 13. Keep the workspace tidy

Every inline query leaves a directory in the workspace holding the SQL it ran
and the CSV it produced. They are durable on purpose — that is what lets you
come back to a Job — so clean up when you are done. `cleanup()` removes
directories older than seven days by default; pass `older_than_days=0` to
clear everything from this session.

In [ ]:
print("workspace:", d.workspace)
print("query dirs:", len(list(d.workspace.iterdir())))
print(d.cleanup(older_than_days=7))     # nothing yet: today's work is younger
print(d.cleanup(older_than_days=0))     # clear the tutorial's results

Cleanup removes results, not history: the Jobs themselves, their manifests and
run logs are untouched, so `d.jobs()` still lists them. Reading a cleaned-up
Result raises `MissingResultError`, which is the honest answer — rerun the Job
or export the table again.

In [ ]:
print("still listed:", any(item.id == job.id for item in d.jobs()))
try:
    d.job(job.id).to_df()
except MissingResultError as exc:
    print(f"MissingResultError: {exc}")

## 14. Habits that keep you out of trouble

- **Filter and limit while exploring.** `d.table(..., limit=1000)` and a
  `WHERE` on the partition column cost minutes instead of hours.
- **Don't loop without `wait_for_slot`.** Two Jobs is the ceiling; a bare loop
  hits it on the third iteration.
- **Materialise once, read many.** If several cells need the same aggregate,
  write it to a table with `destination="Table"` and read that.
- **Keep the Job ID.** It is the only thing you need to recover work after a
  kernel restart, and the thing to quote when asking for help.
- **Read the log before rerunning.** `job.logs()` usually names the real
  problem — a missing table, a memory limit, a queue that stayed full.
- **Take Advisor findings seriously.** Acknowledge them when you mean to, not
  reflexively.
- **Copy deliverables out.** `job.to_csv(path)` for anything that should
  survive `cleanup()`.

## Where to go next

| Topic | Where |
|---|---|
| Full API reference | `docs/notebook-api.md` |
| The same operations in a terminal | `dispatch job --help`, README |
| The interactive TUI | run `dispatch` from your SQL directory |
| Why the API is shaped this way | `docs/adr/0008`–`0011` |
| Query optimisation rules the Advisor checks | `docs/query-optimization-advisor-spec.md` |

One last cell to leave the tutorial's leftovers behind.

In [ ]:
for leftover in ("july_events.csv", "tutorial_query.sql", "tutorial_from_file.csv"):
    (workdir / leftover).unlink(missing_ok=True)

print("workspace directories left:", len(list(d.workspace.iterdir())))
print("Jobs still on record:      ", len(d.jobs()))
print("tables created:            ", sorted(
    job.destination_detail for job in d.jobs() if job.destination == "Table"
))